# Preparación entorno 

Python

In [2]:
!python --version

Python 3.12.0


Dependencias

In [4]:
!Poetry show fastapi

 name         : fastapi                                                                                
 version      : 0.135.1                                                                                
 description  : FastAPI framework, high performance, easy to learn, fast to code, ready for production 

dependencies
 - annotated-doc >=0.0.2
 - email-validator >=2.0.0
 - fastapi-cli >=0.0.8
 - httpx >=0.23.0,<1.0.0
 - jinja2 >=3.1.5
 - pydantic >=2.7.0
 - pydantic-extra-types >=2.0.0
 - pydantic-settings >=2.0.0
 - python-multipart >=0.0.18
 - starlette >=0.46.0
 - typing-extensions >=4.8.0
 - typing-inspection >=0.4.2
 - uvicorn >=0.12.0


In [5]:
!Poetry show numpy

 name         : numpy                                             
 version      : 2.0.2                                             
 description  : Fundamental package for array computing in Python 

required by
 - pandas requires >=1.26.0
 - scikit-learn requires >=1.19.5
 - scipy requires >=1.26.4,<2.7


In [6]:
!Poetry show pandas

 name         : pandas                                                                  
 version      : 2.2.2                                                                   
 description  : Powerful data structures for data analysis, time series, and statistics 

dependencies
 - numpy >=1.26.0
 - python-dateutil >=2.8.2
 - pytz >=2020.1
 - tzdata >=2022.7


In [7]:
!Poetry show scikit-learn

 name         : scikit-learn                                                 
 version      : 1.6.1                                                        
 description  : A set of python modules for machine learning and data mining 

dependencies
 - joblib >=1.2.0
 - numpy >=1.19.5
 - scipy >=1.6.0
 - threadpoolctl >=3.1.0


In [8]:
!Poetry show dill

 name         : dill                    
 version      : 0.3.8                   
 description  : serialize all of Python 


In [9]:
!Poetry show joblib

 name         : joblib                                       
 version      : 1.5.3                                        
 description  : Lightweight pipelining with Python functions 

required by
 - scikit-learn requires >=1.2.0


---
<sup>*Notebook desarrollado por Benjamín Pérez Briceño.*</sup>

El desarrollo de este notebook fue realizado bajo el estandar **CRISP-DM** *(Cross-Industry Standard Process for Data Mining).*

# Fase 1 - Business understanding

## Contexto

**1. Data de Inferencia:** Este archivo *“claims_dataset.csv”* contendrá registros de siniestros de pérdida parcial,
incluyendo la data respectiva para cada variable. Estos son los registros y valores que la API recibe para la
predicción.

**2. Modelo serializado:** Te proporcionaremos seis archivos `.pkl`, cada uno correspondiente a diferentes pasos
del modelo.

**3. Instructivo:** El archivo *“documentación.md”* describe las variables incluidas en los datos de inferencia, las
interdependencias entre los pasos del modelo serializado y detalla las columnas necesarias para realizar las
predicciones. Además, proporciona información clave para el desarrollo del desafío.

## Columnas dataset

- `claim_id`: Número de identificación del siniestro.
- `marca_vehiculo`: Marca del vehículo siniestrado.
- `antiguedad_vehiculo`: Antiguedad del vehículo en años desde su año de fabricación hasta el año del siniestro.
- `tipo_poliza`: Identificador del tipo de póliza asociada al vehículo siniestrado.
- `taller`: Identificador del taller donde se realizarán las reparaciones del vehículo.
- `partes_a_reparar`: Número de partes del vehículo que necesitan reparación.
- `partes_a_reemplazar`: Número de partes del vehículo que deben ser reemplazadas.

# Fase 2 - Data understanding

## Imports

In [ ]:
#Imports
import pandas as pd
import numpy as np
import time

#sklearn
import sklearn

#sklearn - pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer

#sklearn - ml model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

#joblib
import joblib
from joblib import dump, load

#dill
import dill

## Carga de dataset

In [ ]:
#Cargamos dataset, corregimos formato y visualizamos las primeras 5 filas.

df_claims = pd.read_csv('claims_dataset.csv',
                        sep='|',
                        on_bad_lines='warn')
df_claims['valor_vehiculo'] = np.nan

df_claims.head(10)

,claim_id,marca_vehiculo,antiguedad_vehiculo,tipo_poliza,taller,partes_a_reparar,partes_a_reemplazar,valor_vehiculo
0,561205,ferd,1,1,4,3,2,NaN
1,528569,ferd,4,3,1,2,4,NaN
2,408550,NaN,2,4,1,4,2,NaN
3,208451,NaN,1,1,4,1,4,NaN
4,394115,ferd,2,4,1,3,4,NaN
5,243530,fait,2,1,3,1,2,NaN
6,288203,NaN,3,3,4,2,2,NaN
7,991981,ferd,3,3,2,1,2,NaN
8,435094,chepy,2,1,1,4,3,NaN
9,417848,fait,4,3,1,4,4,NaN


## Tipo de dato por columna

In [ ]:
#Visualizamos los tipos de datos de cada columna
df_claims.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   claim_id             10 non-null     int64  
 1   marca_vehiculo       7 non-null      object 
 2   antiguedad_vehiculo  10 non-null     int64  
 3   tipo_poliza          10 non-null     int64  
 4   taller               10 non-null     int64  
 5   partes_a_reparar     10 non-null     int64  
 6   partes_a_reemplazar  10 non-null     int64  
 7   valor_vehiculo       0 non-null      float64
dtypes: float64(1), int64(6), object(1)
memory usage: 772.0+ bytes


# Fase 3 - Data preparation

## Valores nulos y/o duplicados

In [ ]:
#Realizamos una limpieza de datos nulos (NaN/Null)
#df_claims = df_claims.dropna()

#Validamos que no existan valores nulos
print('Valores nulos:')
print(df_claims.isnull().sum())
print('-------------------------')

#Validamos que no existan valores duplicados
print('Valores duplicados:')
print(df_claims.duplicated().sum())

#Podemos ver que existen nulos sin embargo estos no se eliminan ya que más adelante el 'diccionario de imputación' los maneja.

Valores nulos:
claim_id                0
marca_vehiculo          3
antiguedad_vehiculo     0
tipo_poliza             0
taller                  0
partes_a_reparar        0
partes_a_reemplazar     0
valor_vehiculo         10
dtype: int64
-------------------------
Valores duplicados:
0


## Pipelines de Prepocesamiento

El pipeline del modelo, creado y entregado por un Data Scientist, consta de cinco archivos `.pkl` que ejecutan diferentes tareas sobre un dataframe, creando nuevas columnas. Estas son sus interdependencias:

Se desarrollara un pipeline el cual esta compuesto por cinco archivos.

- 1. **pipeline_1.pkl**: Independiente.

- 2. **pipeline_2.pkl**: Dependiente de *pipeline_1.pkl*.

- 3. **pipeline_3.pkl**: Independiente.

- 4. **pipeline_4.pkl**: Dependiente de *pipeline_2.pkl*.

- 5. **pipeline_5.pkl**: Dependiente de *pipeline_3.pkl*.

Se desarrollara un pipeline a partir de un diccionario de inputación de datos.

- 6. **pipeline_6.pkl**: Independiente.

## Descripción de pipelines

pipeline 1 - Se crea una nueva columna `'total_piezas'` la cual se compone de la suma de `'partes a reparar'` y `'partes a reemplazar'`

pipeline 2 - Se utiliza el algoritmo **np.log** para normalizar los valores de las columnas.

pipeline 3 - Crea una nueva columna `'marca_vehiculo_enconded'` creada a partir de `'marca_vehiculo'` para pasar de texto a valores númericos y se pueda procesar la información correctamente.

pipeline 4 - Crea una nueva columna `'valor por pieza'` a partir de `'valor vehiculo'` y `'total piezas'`

## Reconstrucción de pipelines

*(archivos `.pkl`)*

Se tuvo que realizar una reconstrucción de los pipelines a partir de su contenido ya que estos no podian ser leidos ni procesados en su formato actual.

Para estos se reconstruyeron a partir de Scikit learn Pipelines.

### **Funciones**

In [ ]:
#Función n1 - Crea la columna 'total piezas'

def funcion_total_piezas(df_claims):
    df_claims = df_claims.copy()
    df_claims['total_piezas'] = df_claims['partes_a_reparar'] + df_claims['partes_a_reemplazar']
    return df_claims

In [ ]:
#Función n2 - Normalizar valores númericos de la variable anterior 'total piezas'
#(Depende del pipeline n1)

def funcion_normalizar_total_piezas(df_claims):
    df_claims = df_claims.copy()
    time.sleep(2)
    df_claims['log_total_piezas'] = np.log(df_claims['total_piezas'])
    return df_claims

In [ ]:
#Función n3 - Crea la columna 'marca_vehiculos_enconded'
#Creamos un diccionario a partir de las marcas existentes.
#De esta manera convertimos texto a número para ser procesados por el modelo.

def funcion_marca_vehiculos_encoded(df_claims):
  df_claims = df_claims.copy()

  dict_marcas = {
      'chepy':1,
      'fait':2,
      'ferd':3,
  }
  df_claims['marca_vehiculo_encoded'] = df_claims['marca_vehiculo'].map(dict_marcas)
  return df_claims

In [ ]:
#Función n4 - Crea la columna 'valor_por_pieza' y 'valor_vehiculo'
#Se define el valor por vehiculo segun la marca como se definio anteriormente
#Marcas:
# chepy: 1
# fait : 2
# ferd': 3

#Se define el valor por pieza segun el taller
# taller 1: 50
# taller 2: 100
# taller 3: 200
# taller 4: 300
# taller 5: 400

def funcion_valor_vehiculo_y_pieza(df_claims):
  df_claims = df_claims.copy()

  dict_marca = {
      1: 1500,
      2: 2950,
      3: 8540,
  }

  dict_taller = {
      1: 50,
      2: 100,
      3: 200,
      4: 300,
      5: 400
  }

  df_claims['valor_vehiculo'] = df_claims['marca_vehiculo_encoded'].map(dict_marca)
  df_claims['valor_por_pieza'] = df_claims['taller'].map(dict_taller)
  return df_claims


In [ ]:
#Funcion n5 - Crea diccionario de marca y taller
# Se crea las columnas 'reclamos_por_marca' y 'reclamos_por_taller'
# Se añade un time.sleep de 10 segundos

def funcion_reclamos(df_claims):
  df_claims = df_claims.copy()

  diccionario_marca = {
         0: 691,
         1: 691,
         2: 782,
         3: 453
  }

  diccionario_taller = {
        1: 50,
        2: 100,
        3: 200,
        4: 300,
        5: 400
  }

  time.sleep(10)
  df_claims['reclamos_por_marca'] = df_claims['marca_vehiculo_encoded'].map(diccionario_marca)
  df_claims['reclamos_por_taller'] = df_claims['taller'].map(diccionario_taller)
  return df_claims




In [ ]:
#Función n6 - Diccionario de imputación
#Usamos 'fillna()' para imputar en los valores en el dataframe
#Definimos diccionario de imputación

def funcion_diccionario_imputacion(df_claims):
  df_claims = df_claims.copy()


  diccionario_imputacion = {
    'log_total_piezas': 1.4545,
    'marca_vehiculo_encoded': 0,
    'valor_vehiculo': 3560,
    'valor_por_pieza': 150,
    'antiguedad_vehiculo': 1,
    'tipo_poliza': 1,
    'taller': 1,
    'partes_a_reparar': 3,
    'partes_a_reemplazar': 1
  }

  df_claims = df_claims.fillna(diccionario_imputacion)
  return df_claims


### Transformers

In [ ]:
#Creamos los Transformers para cada función
#Se mantiene el mismo orden logico y secuencial de las funciones

#Transformers
transformer_1 = FunctionTransformer(funcion_total_piezas)
transformer_2 = FunctionTransformer(funcion_normalizar_total_piezas)
transformer_3 = FunctionTransformer(funcion_marca_vehiculos_encoded)
transformer_4 = FunctionTransformer(funcion_valor_vehiculo_y_pieza)
transformer_5 = FunctionTransformer(funcion_reclamos)
transformer_6 = FunctionTransformer(funcion_diccionario_imputacion)

### Construimos pipelines

In [ ]:
#Construimos pipelines (Nombre de paso + Transformer)
#Pipeline 1 - Independiente
#Pipeline 2 - Dependiente de pipeline 1
#Pipeline 3 - Independiente
#Pipeline 4 - Dependiente de pipeline 2
#Pipeline 5 - Dependiente de pipeline 3
#Pipeline 6 - Independiente

pipeline_1 = Pipeline([('paso1_total_piezas', transformer_1)])

pipeline_2 = Pipeline([('paso2_normalizar', transformer_2)])

pipeline_3 = Pipeline([('paso3_marca_vehiculos_encoded', transformer_3)])

pipeline_4 = Pipeline([('paso4_valor_por_pieza', transformer_4)])

pipeline_5 = Pipeline([('paso5_reclamos', transformer_5)])

pipeline_6 = Pipeline([('paso_6_imputacion', transformer_6)])

### Carga de pipelines

Cargamos los pipelines de manera individual.Se deben ejecutar de manera secuencial para su correcta aplicación y funcionamiento.

Creamos un nuevo dataframe **(df_modificado)** para aplicar los cambios de los pipelines y evitar modificar el dataframe original.

In [ ]:
#Ya creados los pipelines los cargamos en el nuevo dataset.

df_modificado = pipeline_1.fit_transform(df_claims) #creamos 'total_piezas'

df_modificado = pipeline_3.fit_transform(df_modificado) #creamos marca_vehiculo_encoded

df_modificado = pipeline_2.fit_transform(df_modificado) #creamos log_total_piezas

df_modificado = pipeline_4.fit_transform(df_modificado) #creamos valor_vehiculo y valor_por_pieza

df_modificado = pipeline_5.fit_transform(df_modificado) #creamos reclamos_por_marca y reclamos_por_taller

df_modificado = pipeline_6.fit_transform(df_modificado) #aplicamos el diccionario de imputación

## Dataframe modificado

In [ ]:
#Visualizamos las primeras filas del nuevo dataframe (df_modificado) tras haber aplicado los cambios.
print(df_modificado.head())

   claim_id marca_vehiculo  antiguedad_vehiculo  tipo_poliza  taller  \
0    561205           ferd                    1            1       4   
1    528569           ferd                    4            3       1   
2    408550            NaN                    2            4       1   
3    208451            NaN                    1            1       4   
4    394115           ferd                    2            4       1   

   partes_a_reparar  partes_a_reemplazar  valor_vehiculo  total_piezas  \
0                 3                    2          8540.0             5   
1                 2                    4          8540.0             6   
2                 4                    2          3560.0             6   
3                 1                    4          3560.0             5   
4                 3                    4          8540.0             7   

   marca_vehiculo_encoded  log_total_piezas  valor_por_pieza  \
0                     3.0          1.609438              3

### Valores nulos y/o duplicados

In [ ]:
#Validamos que no existan valores nulos
print('Valores nulos:')
print(df_modificado.isnull().sum())
print('-------------------------')

#Validamos que no existan valores duplicados
print('Valores duplicados:')
print(df_modificado.duplicated().sum())

#A pesar de que existen estos seran inputados más adelante

Valores nulos:
claim_id                  0
marca_vehiculo            3
antiguedad_vehiculo       0
tipo_poliza               0
taller                    0
partes_a_reparar          0
partes_a_reemplazar       0
valor_vehiculo            0
total_piezas              0
marca_vehiculo_encoded    0
log_total_piezas          0
valor_por_pieza           0
reclamos_por_marca        3
reclamos_por_taller       0
dtype: int64
-------------------------
Valores duplicados:
0


# Fase 4 - Modeling

## Modelo de regresión lineal



El modelo de regresión lineal entrenado con data historica devuelve el tiempo en semanas que tardaria un vehiculo en entrar y salir del taller mecánico.

El modelo necesaria de las siguentes variables para para predecir:

- `'log_total_piezas'`: float64
- `'marca_vehiculo_encoded'`: int64
- `'valor_vehiculo'`: int64
- `'valor_por_pieza'`: int64
- `'antiguedad_vehiculo'`: int64


### Cargar modelo

In [ ]:
#Cargamos modelo linear de regresión en pkl
modelo_RL = joblib.load('linnear_regression.pkl')

#Advertencia al ejecutar: El modelo fue creado bajo la versión de sklearn 1.3.0 mientras la versión actual del entorno (Colab) corresponde a 1.6.1.

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
#Validamos que el modelo funcione una vez cargado
print(type(modelo_RL))

<class 'sklearn.linear_model._base.LinearRegression'>


### Features de predicción

In [ ]:
#Por 'Features' se entienden las columnas que el modelo de regresión necesita para predecir.
#Definimos features
FEATURES_MODELO = [
    'log_total_piezas',
    'marca_vehiculo_encoded',
    'valor_vehiculo',
    'valor_por_pieza',
    'antiguedad_vehiculo'
]

In [ ]:
#Definimos el tipo de dato que debe llevar a cada feature (float64/int64) en 'X'.
#Verificamos que se apliquen correctamente

X = df_modificado[FEATURES_MODELO].astype({
    'log_total_piezas': 'float64',
    'marca_vehiculo_encoded': 'int64',
    'valor_vehiculo': 'int64',
    'valor_por_pieza': 'int64',
    'antiguedad_vehiculo': 'int64'
})

print('Tipo de dato de FEATURES_MODELO:')
print('--------------------------------')
print(X.dtypes)
print('--------------------------------')
print('Visualizamos las primeras 5 filas')
X.head(5)

Tipo de dato de FEATURES_MODELO:
--------------------------------
log_total_piezas          float64
marca_vehiculo_encoded      int64
valor_vehiculo              int64
valor_por_pieza             int64
antiguedad_vehiculo         int64
dtype: object
--------------------------------
Visualizamos las primeras 5 filas


,log_total_piezas,marca_vehiculo_encoded,valor_vehiculo,valor_por_pieza,antiguedad_vehiculo
0,1.609438,3,8540,300,1
1,1.791759,3,8540,50,4
2,1.791759,0,3560,50,2
3,1.609438,0,3560,300,1
4,1.945910,3,8540,50,2


### Predicción del modelo

Construimos la predicción del modelo a partir del formato requerido y las condiciones especiales fijadas.

- El cliente que usara el modelo predictivo expreso que cuando el tipo de poliza es igual a 4,el modelo debe devolver un '-1' ya que ese tipo de polizas se manejan de forma especial.

In [ ]:
#Definimos el modelo en 'predicciones_RL' para predecir las Features
predicciones_RL = modelo_RL.predict(X)

#### **Condición poliza**

In [ ]:
# Dado que dentro de las features con las que trabaja el modelo no estan consideradas las siguentes variables:
# - 'tipo_poliza' -> (Para cumplir la condición declarada por el cliente)
# - 'claim_id'    -> (Para identificar el id de la solicitud de siniestro)

#Hacemos copia desde el 'dataset_modificado'
df_condicion_poliza = df_modificado[['claim_id', 'tipo_poliza']].copy()

In [ ]:
#Creamos una nueva columna 'prediccion_semanas' en la cual iran la predicciones de las features
#Forma parte del valor que se modificara más adelante bajo la condición del cliente.

df_condicion_poliza['prediccion_semanas'] = predicciones_RL

In [ ]:
#Validamos que se crea la nueva columna y esta poblada
df_condicion_poliza.head()

,claim_id,tipo_poliza,prediccion_semanas
0,561205,1,4.384084
1,528569,3,2.949009
2,408550,4,5.617430
3,208451,1,6.165576
4,394115,4,3.506305


In [ ]:
#Aplicamos la condición declarada por el cliente
#Usamos 'df.loc' para identificar primero si la columna 'tipo_poliza' es 4, la nueva columna 'predicción semanas' pasa a ser valor '-1'

df_condicion_poliza.loc[df_condicion_poliza['tipo_poliza'] == 4, 'prediccion_semanas'] = -1

#### **Prediccion final**

In [ ]:
#Validamos nuevamente que cumpla la condición (Si 'tipo_poliza' = 4 -> 'prediccion_semanas' = -1)
df_condicion_poliza.head()

,claim_id,tipo_poliza,prediccion_semanas
0,561205,1,4.384084
1,528569,3,2.949009
2,408550,4,-1.000000
3,208451,1,6.165576
4,394115,4,-1.000000


# Fase 5 - Evaluation

# Fase 6 - Deployment

### Exportar pipelines

Usamos **joblib** para exportar los pipelines como objetos.

*(Una vez ejecutada la celda de código los pipelines quedaran guardados como objetos en el directorio actual.)*

#### Pipelines individuales

In [ ]:
#Exportamos con 'joblib.dump(archivo, 'nombre_archivo')
#Posteriormente podemos usar joblib.load('nombre_archivo') -> ej:  joblib.load('pipeline_1.joblib') para validar.

joblib.dump(pipeline_1, 'pipeline_1.joblib')
joblib.dump(pipeline_2, 'pipeline_2.joblib')
joblib.dump(pipeline_3, 'pipeline_3.joblib')
joblib.dump(pipeline_4, 'pipeline_4.joblib')
joblib.dump(pipeline_5, 'pipeline_5.joblib')
joblib.dump(pipeline_6, 'pipeline_6.joblib')

['pipeline_6.joblib']

In [ ]:
#Quitar comentario para probar pipelines de manera individual.
#joblib.load('pipeline_1.joblib')

#### Pipeline completo

Dado que los pipelines seran consumidos por la API se decide encapsular en un solo pipeline.

In [ ]:
#Para la construccioon del pipeline completo llamamos ocupamos los Transformers creados anteriormente

pipeline_completo = Pipeline([
    ('paso1_total_piezas',    FunctionTransformer(funcion_total_piezas)),
    ('paso3_marca_encoded',   FunctionTransformer(funcion_marca_vehiculos_encoded)),
    ('paso2_log_norm',        FunctionTransformer(funcion_normalizar_total_piezas)),
    ('paso4_valor',           FunctionTransformer(funcion_valor_vehiculo_y_pieza)),
    ('paso5_reclamos',        FunctionTransformer(funcion_reclamos)),
    ('paso6_imputacion',      FunctionTransformer(funcion_diccionario_imputacion)),
])

In [ ]:
#Verificamos que el pipeline_completo funciona correctamente
df_prueba = pipeline_completo.fit_transform(df_claims)
df_prueba.head()

,claim_id,marca_vehiculo,antiguedad_vehiculo,tipo_poliza,taller,partes_a_reparar,partes_a_reemplazar,valor_vehiculo,total_piezas,marca_vehiculo_encoded,log_total_piezas,valor_por_pieza,reclamos_por_marca,reclamos_por_taller
0,561205,ferd,1,1,4,3,2,8540.0,5,3.0,1.609438,300,453.0,300
1,528569,ferd,4,3,1,2,4,8540.0,6,3.0,1.791759,50,453.0,50
2,408550,NaN,2,4,1,4,2,3560.0,6,0.0,1.791759,50,NaN,50
3,208451,NaN,1,1,4,1,4,3560.0,5,0.0,1.609438,300,NaN,300
4,394115,ferd,2,4,1,3,4,8540.0,7,3.0,1.945910,50,453.0,50


In [ ]:
#Se exporta el pipeline completo (creado a partir de Transformers)
#joblib.dump (pipeline_completo ,'pipeline_completo.joblib')

['pipeline_completo.joblib']

### Exportar modelo de regresión lineal

In [ ]:
#Se exporta el modelo final en .pkl/.joblib para que sea
#procesado por la API. (FastAPI)
#joblib.dump(modelo_RL, 'modelo_RL.joblib')

['modelo_RL.joblib']

In [ ]:
# Verificación del flujo completo
pipeline_cargado = joblib.load('pipeline_completo.joblib')
modelo_cargado   = joblib.load('modelo_RL.joblib')

df_verificacion  = pipeline_cargado.fit_transform(df_claims)
X_verificacion   = df_verificacion[FEATURES_MODELO].astype({
    'log_total_piezas':       'float64',
    'marca_vehiculo_encoded': 'int64',
    'valor_vehiculo':         'int64',
    'valor_por_pieza':        'int64',
    'antiguedad_vehiculo':    'int64'
})

preds = modelo_cargado.predict(X_verificacion)
print('Verificacion OK — predicciones:', preds.round(2))

Verificacion OK — predicciones: [4.38 2.95 5.62 6.17 3.51 6.04 5.76 4.82 4.98 3.23]


# Exportamos con dill

Para efectos de desarrollo, consumo del modelo y los pipelines para la API se exportan los artefactos con 'dill'

In [ ]:
#Posteriormente se mueven a las carpetas de desarrollo para que la API los consuma

import dill

#Pipelines
with open ('pipeline_completo','wb') as f:
    dill.dump(pipeline_completo, f)

#Modelo RL
with open ('modelo_RL','wb') as f:
    dill.dump(modelo_RL, f)



# FastAPI app

Se desarrollara una aplicación en FastAPI que se consumira el modelo de regresión lineal para predecir las features definidas.

El modelo necesita de las siguientes features para predecir:

**Nombre Columna** | **Formato del Campo en Entrenamiento**
- 'log_total_piezas': float64
- 'marca_vehiculo_encoded': int64
- 'valor_vehiculo': int64
- 'valor_por_pieza': int64
- 'antiguedad_vehiculo': int64
---
**Repositorio GitHub**: *adjuntar link*

In [ ]:
#ADJUNTAR LINK DE REPOSITORIO GITHUB UNA VEZ DESARROLLADO, CREADO Y FUNCIONANDO

In [ ]:
import joblib
import pandas as pd
import numpy as np

# Cargamos ambos artefactos
pipeline_cargado = joblib.load('pipeline_completo.joblib')
modelo_cargado   = joblib.load('modelo_RL.joblib')

# Verificamos tipos
print('Tipo pipeline:', type(pipeline_cargado))
print('Tipo modelo:  ', type(modelo_cargado))
print()

# Verificamos steps del pipeline
print('Steps del pipeline:')
for nombre, step in pipeline_cargado.steps:
    print(f'  {nombre}: {type(step).__name__}')
print()

# Verificamos atributos del modelo
print('Tipo modelo interno:  ', type(modelo_cargado).__name__)
print('Coeficientes:         ', modelo_cargado.coef_.round(6))
print('Intercepto:           ', round(modelo_cargado.intercept_, 6))
print('Features esperadas:   ', modelo_cargado.n_features_in_)

Tipo pipeline: <class 'sklearn.pipeline.Pipeline'>
Tipo modelo:   <class 'sklearn.linear_model._base.LinearRegression'>

Steps del pipeline:
  paso1_total_piezas: FunctionTransformer
  paso3_marca_encoded: FunctionTransformer
  paso2_log_norm: FunctionTransformer
  paso4_valor: FunctionTransformer
  paso5_reclamos: FunctionTransformer
  paso6_imputacion: FunctionTransformer

Tipo modelo interno:   LinearRegression
Coeficientes:          [-2.138387e+00 -4.654910e-01 -7.700000e-05 -1.141000e-03 -4.434650e-01]
Intercepto:            10.668109
Features esperadas:    5
